# Objective 4 — Forecast weekly orders (Linear Regression)

**Goal:** predict how many orders will be placed next week, using what happened in previous weeks.

**Data:** `weekly_orders.csv`, exported from the data warehouse **TokenmafiaDW(final)** (`Fact_Orders` joined to `Dim_Date` on the purchase date).
One row = one week (weeks start on Monday). Columns:
- `week_start`: Monday of the week
- `n_orders`: number of orders placed that week (**this is what we predict**)

> **Note:** the DW only loads delivered orders (plus 6 canceled ones), so the target is really *weekly delivered orders*.

**How to run in Colab:** open the **Files** panel (folder icon on the left), upload `weekly_orders.csv`, then run the cells from top to bottom.

## 1. Import the libraries

In [ ]:
import pandas as pd                  # tables
import numpy as np                   # math
import matplotlib.pyplot as plt      # charts

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

## 2. Load the data

In [ ]:
df = pd.read_csv("weekly_orders.csv", parse_dates=["week_start"])

print("Shape:", df.shape)
df.head()

## 3. Quick checks (cleaning)
The CSV was already cleaned when it was exported:
- weeks before 2017 removed (almost no orders, the platform was just starting)
- the last 2 weeks of August 2018 removed (the data stops in the middle of them, so the counts are too low)
- weeks with no orders filled with 0 so there are no gaps

We still double-check here: no missing values, no duplicated weeks, every week present, and data sorted by date.

In [ ]:
print("Missing values:\n", df.isna().sum(), "\n")
print("Duplicated weeks:", df["week_start"].duplicated().sum())

# Every row should be exactly 7 days after the previous one
gaps = df["week_start"].diff().dropna().dt.days
print("All weeks consecutive:", (gaps == 7).all())

df = df.sort_values("week_start").reset_index(drop=True)
df.describe()

## 4. Look at the data
The big spike in November 2017 is **Black Friday**.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df["week_start"], df["n_orders"], marker="o", markersize=3)
plt.title("Orders per week")
plt.xlabel("Week")
plt.ylabel("Number of orders")
plt.grid(alpha=0.3)
plt.show()

### 4.1 Trend: smoothing with a 4-week average
Weekly numbers are noisy. A 4-week moving average shows the general direction: orders grow through 2017 and stabilise in 2018.

In [ ]:
smooth = df["n_orders"].rolling(4).mean()

plt.figure(figsize=(12, 4))
plt.plot(df["week_start"], df["n_orders"], color="lightgray", label="Weekly orders")
plt.plot(df["week_start"], smooth, color="tab:blue", linewidth=2.5, label="4-week moving average")
plt.title("Trend of weekly orders")
plt.xlabel("Week")
plt.ylabel("Number of orders")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 4.2 Distribution of weekly orders
How many orders does a typical week have? The far-right bar is the Black Friday week.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["n_orders"], bins=20, color="tab:blue", edgecolor="white")
plt.axvline(df["n_orders"].mean(), color="red", linestyle="--", label=f"Mean = {df['n_orders'].mean():.0f}")
plt.axvline(df["n_orders"].median(), color="orange", linestyle="--", label=f"Median = {df['n_orders'].median():.0f}")
plt.title("Distribution of weekly orders")
plt.xlabel("Orders in a week")
plt.ylabel("Number of weeks")
plt.legend()
plt.show()

### 4.3 Orders per week, year by year
Comparing 2017 and 2018 on the same calendar shows the growth from one year to the next.

In [ ]:
plt.figure(figsize=(12, 4))
for year, part in df.groupby(df["week_start"].dt.year):
    plt.plot(part["week_start"].dt.isocalendar().week, part["n_orders"], marker="o", markersize=3, label=str(year))
plt.title("Weekly orders by year")
plt.xlabel("Week of the year")
plt.ylabel("Number of orders")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 4.4 Average orders by month
Careful: most months appear in only one or two years, so this mixes seasonality with growth.

In [ ]:
monthly = df.groupby(df["week_start"].dt.month)["n_orders"].mean()

plt.figure(figsize=(10, 4))
plt.bar(monthly.index, monthly.values, color="tab:blue")
plt.xticks(range(1, 13), ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
plt.title("Average weekly orders by month")
plt.ylabel("Average orders per week")
plt.show()

## 5. Build the features
A regression model needs input columns (features) to predict the target. For a time series, the best clues are the **previous weeks**:

| Feature | Meaning |
|---|---|
| `lag_1` | orders of the previous week |
| `rolling_mean_4` | average orders of the 4 previous weeks |
| `week_of_year` | week number (1–52) |
| `month` | month number (1–12) |
| `black_friday_flag` | 1 if the week contains Black Friday, else 0 |

**Important:** `shift(1)` moves values down one row, so each week only sees **past** weeks. Without it the model would see the answer (data leakage).

In [ ]:
# Values from the past
df["lag_1"] = df["n_orders"].shift(1)
df["rolling_mean_4"] = df["n_orders"].shift(1).rolling(4).mean()

# Calendar
df["week_of_year"] = df["week_start"].dt.isocalendar().week.astype(int)
df["month"] = df["week_start"].dt.month

# Black Friday = 4th Friday of November. Our week starts on Monday, so the Friday is 4 days later.
def is_black_friday_week(week_start):
    friday = week_start + pd.Timedelta(days=4)
    return int(friday.month == 11 and 22 <= friday.day <= 28)

df["black_friday_flag"] = df["week_start"].apply(is_black_friday_week)

df[df["black_friday_flag"] == 1]

The first 4 weeks have no full history yet (empty `lag_1` / `rolling_mean_4`), so we drop them.

In [ ]:
df = df.dropna().reset_index(drop=True)
print("Rows left:", len(df))
df.head()

### 5.1 Does last week predict this week?
Each dot is one week: x = orders of the previous week, y = orders of this week.
If the dots follow the diagonal, `lag_1` is a good predictor. The red dot is Black Friday (far above the line: last week could not predict it).

In [ ]:
bf = df["black_friday_flag"] == 1

plt.figure(figsize=(6, 6))
plt.scatter(df.loc[~bf, "lag_1"], df.loc[~bf, "n_orders"], alpha=0.7, label="Normal week")
plt.scatter(df.loc[bf, "lag_1"], df.loc[bf, "n_orders"], color="red", s=80, label="Black Friday week")
lims = [0, df["n_orders"].max() * 1.05]
plt.plot(lims, lims, color="gray", linestyle="--", label="this week = last week")
plt.xlim(lims); plt.ylim(lims)
plt.title("Orders this week vs previous week")
plt.xlabel("lag_1 (orders last week)")
plt.ylabel("n_orders (orders this week)")
plt.legend()
plt.show()

### 5.2 Correlation between features and target
Correlation goes from -1 to 1. Close to 1 = when the feature goes up, orders go up too.
The bottom row (`n_orders`) is the one that matters: `lag_1` and `rolling_mean_4` are strongly linked to the target.
Also notice that `week_of_year` and `month` are almost perfectly correlated with each other (they carry the same information), which is why the model later gives them large opposite coefficients.

In [ ]:
cols = ["lag_1", "rolling_mean_4", "week_of_year", "month", "black_friday_flag", "n_orders"]
corr = df[cols].corr()

plt.figure(figsize=(7, 6))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(cols)), cols, rotation=45, ha="right")
plt.yticks(range(len(cols)), cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

## 6. Split into train and test (by time)
With time series we **never shuffle**: we train on the past and test on the most recent weeks, like a real forecast.
We keep the **last 12 weeks** for testing.

In [ ]:
features = ["lag_1", "rolling_mean_4", "week_of_year", "month", "black_friday_flag"]
target = "n_orders"

test_weeks = 12
train = df.iloc[:-test_weeks]
test = df.iloc[-test_weeks:]

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

print("Train:", train["week_start"].min().date(), "->", train["week_start"].max().date(), f"({len(train)} weeks)")
print("Test :", test["week_start"].min().date(), "->", test["week_start"].max().date(), f"({len(test)} weeks)")

The chart shows which weeks the model learns from (blue) and which ones we hide to test it (orange).

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(train["week_start"], train["n_orders"], color="tab:blue", label="Train (69 weeks)")
plt.plot(test["week_start"], test["n_orders"], color="tab:orange", label="Test (12 weeks)")
plt.axvline(test["week_start"].iloc[0], color="red", linestyle=":")
plt.title("Train / test split")
plt.xlabel("Week")
plt.ylabel("Number of orders")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 7. Baseline: "next week = this week"
Before training a model we need something to compare it to. The simplest guess is that next week will have the same number of orders as last week (`lag_1`).
The model is only useful if it beats this.

In [ ]:
def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)                 # average error in orders
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))         # punishes big errors more
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100   # average error in %
    return {"model": name, "MAE": round(mae, 1), "RMSE": round(rmse, 1), "MAPE %": round(mape, 1)}

baseline_pred = X_test["lag_1"]
results = [evaluate("Baseline (last week)", y_test, baseline_pred)]
pd.DataFrame(results)

## 8. Train the Linear Regression
Linear regression learns one weight per feature:

`n_orders ≈ b0 + b1·lag_1 + b2·rolling_mean_4 + b3·week_of_year + b4·month + b5·black_friday_flag`

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

lr_pred = model.predict(X_test)
results.append(evaluate("Linear Regression", y_test, lr_pred))
pd.DataFrame(results)

## 9. What did the model learn?
Each coefficient tells how much the prediction changes when that feature goes up by 1.
For example, a `lag_1` coefficient of 0.5 means "+100 orders last week → about +50 orders predicted".

In [ ]:
coefs = pd.DataFrame({"feature": features, "coefficient": model.coef_.round(3)})
print("Intercept (b0):", round(model.intercept_, 1))
coefs

The same coefficients as a chart. Green = pushes the prediction up, red = pushes it down.
`black_friday_flag` is huge because it is a 0/1 switch for one special week, while `lag_1` is multiplied by values in the thousands, so bar sizes are not directly comparable.

In [ ]:
colors = ["tab:green" if c > 0 else "tab:red" for c in coefs["coefficient"]]

plt.figure(figsize=(8, 4))
plt.barh(coefs["feature"], coefs["coefficient"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Linear Regression coefficients")
plt.xlabel("Change in predicted orders when the feature increases by 1")
plt.tight_layout()
plt.show()

## 10. Plot: actual vs predicted on the test weeks

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df["week_start"], df["n_orders"], label="Actual", color="gray", alpha=0.6)
plt.plot(test["week_start"], lr_pred, label="Linear Regression", marker="o")
plt.plot(test["week_start"], baseline_pred, label="Baseline (last week)", linestyle="--")
plt.axvline(test["week_start"].iloc[0], color="red", linestyle=":", label="Start of test")
plt.title("Weekly orders: actual vs predicted")
plt.xlabel("Week")
plt.ylabel("Number of orders")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 10.1 Zoom on the test weeks
Same chart, only the 12 test weeks. Notice that both predictions arrive **one week late**: when orders jump, the model only sees it the week after.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(test["week_start"], y_test, marker="o", color="black", label="Actual")
plt.plot(test["week_start"], lr_pred, marker="o", label="Linear Regression")
plt.plot(test["week_start"], baseline_pred, marker="o", linestyle="--", label="Baseline (last week)")
plt.title("Test weeks: actual vs predicted")
plt.xlabel("Week")
plt.ylabel("Number of orders")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 10.2 Predicted vs actual
A perfect model would put every dot on the dashed diagonal. Dots above the line = the model predicted too low; below = too high.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, lr_pred, s=60)
lims = [min(y_test.min(), lr_pred.min()) * 0.9, max(y_test.max(), lr_pred.max()) * 1.05]
plt.plot(lims, lims, color="gray", linestyle="--", label="Perfect prediction")
plt.xlim(lims); plt.ylim(lims)
plt.title("Linear Regression: predicted vs actual (test)")
plt.xlabel("Actual orders")
plt.ylabel("Predicted orders")
plt.legend()
plt.show()

### 10.3 Errors (residuals)
Residual = actual - predicted. Positive = the model predicted too few orders, negative = too many.
Good residuals are small, centred on 0 and without a pattern.
Here most bars are positive: the model **tends to predict too few orders** in summer 2018, because orders were higher than the level it learned from.

In [ ]:
residuals = y_test.values - lr_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

bar_colors = ["tab:green" if r > 0 else "tab:red" for r in residuals]
axes[0].bar(test["week_start"].dt.strftime("%m-%d"), residuals, color=bar_colors)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Residual for each test week")
axes[0].set_xlabel("Week (month-day, 2018)")
axes[0].set_ylabel("Actual - predicted")
axes[0].tick_params(axis="x", rotation=45)

axes[1].hist(residuals, bins=8, color="tab:blue", edgecolor="white")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Distribution of residuals")
axes[1].set_xlabel("Actual - predicted")
axes[1].set_ylabel("Number of weeks")

plt.tight_layout()
plt.show()

### 10.4 Model comparison
Lower bars = better. This is the results table from steps 7–8 as a chart.

In [ ]:
res = pd.DataFrame(results).set_index("model")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, ["MAE", "RMSE", "MAPE %"]):
    ax.bar(res.index, res[metric], color=["gray", "tab:blue"])
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=15)
    for i, v in enumerate(res[metric]):
        ax.text(i, v, f"{v}", ha="center", va="bottom")
plt.suptitle("Baseline vs Linear Regression (lower is better)")
plt.tight_layout()
plt.show()

## 11. Forecast the next 4 weeks
The objective asks for a forecast **4 weeks ahead**. We do not know the future orders, so we predict **one week at a time**
and use each prediction as the "last week" for the next one (this is called a *recursive* forecast).

First we retrain the model on **all** the data so it knows the most recent weeks.

In [ ]:
final_model = LinearRegression()
final_model.fit(df[features], df[target])

history = list(df["n_orders"])            # all known weekly orders
last_week = df["week_start"].iloc[-1]
forecast = []

for step in range(1, 5):
    week = last_week + pd.Timedelta(weeks=step)
    row = pd.DataFrame([{
        "lag_1": history[-1],
        "rolling_mean_4": np.mean(history[-4:]),
        "week_of_year": week.isocalendar().week,
        "month": week.month,
        "black_friday_flag": is_black_friday_week(week),
    }])
    pred = final_model.predict(row[features])[0]
    forecast.append({"week_start": week.date(), "predicted_orders": round(pred)})
    history.append(pred)                  # the prediction becomes next week's lag_1

forecast_df = pd.DataFrame(forecast)
forecast_df

### 11.1 Forecast chart
The last 20 known weeks followed by the 4 forecast weeks. The shaded band is ± the test MAE: a rough idea of how wrong the forecast can be.

In [ ]:
recent = df.tail(20)
fc_dates = pd.to_datetime(forecast_df["week_start"])
fc_values = forecast_df["predicted_orders"]
mae = mean_absolute_error(y_test, lr_pred)

plt.figure(figsize=(12, 4))
plt.plot(recent["week_start"], recent["n_orders"], marker="o", color="black", label="Known weeks")
plt.plot([recent["week_start"].iloc[-1], fc_dates.iloc[0]],
         [recent["n_orders"].iloc[-1], fc_values.iloc[0]], color="tab:blue", linestyle="--")
plt.plot(fc_dates, fc_values, marker="o", color="tab:blue", linestyle="--", label="Forecast")
plt.fill_between(fc_dates, fc_values - mae, fc_values + mae, color="tab:blue", alpha=0.15, label=f"± MAE ({mae:.0f})")
plt.title("4-week forecast of weekly orders")
plt.xlabel("Week")
plt.ylabel("Number of orders")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 12. Conclusion
- On the 12 test weeks, the Linear Regression is off by about **220 orders per week (~14%)**, while the "last week" baseline is off by about **196 (~13%)**. So the model is **about as good as the baseline, not better**.
- Why: the test weeks (June–August 2018) jump up and down a lot (for example 972 → 1,649 orders from one week to the next). Nothing in our features explains those jumps, so the best any simple model can do is follow the recent level.
- `lag_1` and `rolling_mean_4` carry most of the signal (positive coefficients: more orders recently → more orders predicted).
- `week_of_year` and `month` have large coefficients with opposite signs. With little more than one year of data they mostly cancel each other out and add noise; removing them brings the error close to the baseline.
- `black_friday_flag` gets a very large coefficient (~+1,850 orders) because it is fitted on a **single** Black Friday. That number should not be trusted.
- **Limits:**
  - only ~80 weeks of data, and only **one** Black Friday, so the model cannot really learn seasonality or holiday effects;
  - the DW contains delivered orders only;
  - recursive forecasts get less reliable the further ahead they go (errors pile up).
- **Possible next steps:** try `Ridge`, `SVR` or `KNeighborsRegressor` with the same train/test split and compare them in the same results table.